TEST COPY — targets are *_TEST / *_test only. Original files are never touched.

In [ ]:
# ================= 自动执行逻辑 (第二阶段：入库) =================
# 必须依赖 Cell 1 成功运行后定义的变量和路径

import os
import concurrent.futures
from threading import Lock

# 定义路径 (重定义关键变量)
CATALOG_NAME = "workspace"
SCHEMA_NAME = "default"
VOLUME_NAME = "course_data_test"
TARGET_CATALOG = "workspace"
TARGET_SCHEMA = "course_db_test"
DATABASE_DATA_VOLUME_FOLDER = "Database_Data" 

volume_base_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"
database_volume_path = f"{volume_base_path}/{DATABASE_DATA_VOLUME_FOLDER}/"

print(f"🔄 正在将 {DATABASE_DATA_VOLUME_FOLDER} 中的文件转为 Delta 表...")

progress_lock = Lock()

def sanitize_column_names(df):
    forbidden_chars = [' ', ',', ';', '{', '}', '(', ')', '\n', '\t', '=']
    new_columns = []
    col_map = {}
    for col in df.columns:
        new_col = col
        for ch in forbidden_chars:
            new_col = new_col.replace(ch, '_')
        # Ensure uniqueness if duplicate after sanitization
        orig_new_col = new_col
        suffix = 1
        while new_col in new_columns:
            new_col = f"{orig_new_col}_{suffix}"
            suffix += 1
        new_columns.append(new_col)
        col_map[col] = new_col
    # Print warning if any columns were renamed
    renamed = [(k, v) for k, v in col_map.items() if k != v]
    if renamed:
        print(f"      ⚠️ 列名已重命名: {renamed}")
    return df.toDF(*new_columns)

def process_file(file_name):
    table_name = file_name.replace('.', '_').replace('-', '_').split('_csv')[0]
    source_file_path = os.path.join(database_volume_path, file_name)
    target_table = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table_name}"
    with progress_lock:
        print(f"   -> 正在创建表: {table_name}")
    try:
        df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .option("delimiter", ",")
              .csv(source_file_path))
        if len(df.columns) == 0:
            with progress_lock:
                print(f"      ⚠️ 文件 {file_name} 没有有效的列，已跳过 (可能是空文件或非数据文件)。")
            return
        df = sanitize_column_names(df)
        # 直接写入 Delta 表，覆盖原表
        df.write.format("delta").mode("overwrite").saveAsTable(target_table)
        with progress_lock:
            print(f"      ✅ Delta 表 {table_name} 已创建并导入成功。")
    except Exception as e:
        with progress_lock:
            print(f"      ❌ 表 {table_name} 导入失败。请检查文件格式是否为 CSV/Parquet/JSON。")
            print(f"         错误: {str(e)}")

# 确保目标 Schema 存在
try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")
    print(f"✅ 目标 Schema 已准备: {TARGET_CATALOG}.{TARGET_SCHEMA}")
except Exception as e:
    print(f"⚠️ Schema 创建失败: {e}")

# 确保目标文件夹存在
if not os.path.exists(database_volume_path):
    print("❌ 转换失败：database_data 文件夹在 Volume 中不存在。请检查 Cell 1 是否成功运行。")
else:
    try:
        file_names = os.listdir(database_volume_path)
    except Exception as e:
        print(f"❌ 访问 Volume 路径失败，请检查集群是否挂载正常。错误: {e}")
        file_names = []
    data_files = [f for f in file_names if not f.startswith('.') and os.path.isfile(os.path.join(database_volume_path, f))]
    if not data_files:
        print("❌ 转换失败：os.listdir 仍然未找到任何文件。请再次手动刷新 Catalog Explorer，并确认文件是否为空。")
    else:
        # 并行处理文件
        with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
            executor.map(process_file, data_files)

print("="*40)
print(f"🎉 第二阶段完成！数据入库成功。")
print(f"请刷新 Catalog Explorer，并在 {TARGET_CATALOG}.{TARGET_SCHEMA} 下查看您的新表格。")